# EMERGENCY DIAGNOSTICS - Find What's Actually Wrong

Training loss is decreasing but validation is random (AUROC ~0.65).

This notebook will find the REAL problem.

---

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os
from pathlib import Path

BASE_DIR = '/content/drive/MyDrive/vindr-mammo'
PREPROCESSED_DIR = f'{BASE_DIR}/preprocessed_png_512'
STRATIFIED_CSV = 'stratified_selection.csv'  # In current directory

print("Loading data...")
df = pd.read_csv(STRATIFIED_CSV)
df['image_path'] = df['file_path'].str.replace('.dicom', '.png', regex=False)
df['patient_id'] = df['study_id']

print(f"Total images: {len(df)}")

## 1. CHECK: Are Images Actually Different?

In [ ]:
print("Checking if images are actually different...\n")

# Load first 10 images
images = []
for idx in range(min(10, len(df))):
    img_path = os.path.join(PREPROCESSED_DIR, df.iloc[idx]['image_path'])
    if os.path.exists(img_path):
        img = np.array(Image.open(img_path))
        images.append(img)

if len(images) >= 2:
    # Check if all images are identical
    all_same = all(np.array_equal(images[0], img) for img in images[1:])
    
    if all_same:
        print("❌ CRITICAL: ALL IMAGES ARE IDENTICAL!")
        print("   Your PNG conversion is broken!")
    else:
        print("✅ Images are different")
        
        # Check diversity
        means = [img.mean() for img in images]
        stds = [img.std() for img in images]
        
        print(f"   Mean pixel values: {np.min(means):.1f} - {np.max(means):.1f}")
        print(f"   Std pixel values: {np.min(stds):.1f} - {np.max(stds):.1f}")
        
        if np.std(means) < 1.0:
            print("   ⚠️  WARNING: Images look very similar!")
else:
    print("❌ Could not load images!")

## 2. CHECK: Visualize Actual Images Being Loaded

In [ ]:
# Show malignant vs benign
mal_sample = df[df['label'] == 1].head(3)
ben_sample = df[df['label'] == 0].head(3)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('ACTUAL LOADED IMAGES (After PNG Conversion)', fontsize=16, fontweight='bold')

for idx, (ax, (_, row)) in enumerate(zip(axes[0], mal_sample.iterrows())):
    img_path = os.path.join(PREPROCESSED_DIR, row['image_path'])
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img, cmap='gray')
        ax.set_title(f"MALIGNANT\n{row['breast_birads']}\nMean: {np.array(img).mean():.1f}", color='red')
    else:
        ax.text(0.5, 0.5, 'NOT FOUND', ha='center', va='center')
    ax.axis('off')

for idx, (ax, (_, row)) in enumerate(zip(axes[1], ben_sample.iterrows())):
    img_path = os.path.join(PREPROCESSED_DIR, row['image_path'])
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img, cmap='gray')
        ax.set_title(f"BENIGN\n{row['breast_birads']}\nMean: {np.array(img).mean():.1f}", color='green')
    else:
        ax.text(0.5, 0.5, 'NOT FOUND', ha='center', va='center')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n⚠️  LOOK AT THE IMAGES ABOVE:")
print("   - Are they all black/white/blank?")
print("   - Can you see breast tissue?")
print("   - Do malignant look different from benign?")

## 3. CRITICAL CHECK: Label Distribution

In [ ]:
print("="*70)
print("LABEL ANALYSIS")
print("="*70)

print(f"\nTotal images: {len(df)}")
print(f"Malignant (1): {(df['label'] == 1).sum()}")
print(f"Benign (0): {(df['label'] == 0).sum()}")

print(f"\nBI-RADS → Label mapping:")
for birads in df['breast_birads'].unique():
    subset = df[df['breast_birads'] == birads]
    mal = (subset['label'] == 1).sum()
    ben = (subset['label'] == 0).sum()
    print(f"  {birads}: {mal} malignant, {ben} benign")

# CRITICAL: Check if labels make sense
birads_4_5_benign = df[(df['breast_birads'].isin(['BI-RADS 4', 'BI-RADS 5'])) & (df['label'] == 0)]
birads_1_2_malignant = df[(df['breast_birads'].isin(['BI-RADS 1', 'BI-RADS 2'])) & (df['label'] == 1)]

if len(birads_4_5_benign) > 0:
    print(f"\n❌ CRITICAL: {len(birads_4_5_benign)} BI-RADS 4/5 labeled as BENIGN!")
    print(f"   BI-RADS 4/5 should be MALIGNANT!")
    print(f"   YOUR LABELS ARE WRONG!")

if len(birads_1_2_malignant) > 0:
    print(f"\n❌ CRITICAL: {len(birads_1_2_malignant)} BI-RADS 1/2 labeled as MALIGNANT!")
    print(f"   BI-RADS 1/2 should be BENIGN!")
    print(f"   YOUR LABELS ARE WRONG!")

print("\n" + "="*70)

## 4. CHECK: Train/Val Split Makes Sense?

In [ ]:
# Recreate split
patients = df['patient_id'].unique()
np.random.seed(42)
np.random.shuffle(patients)

n_train = int(len(patients) * 0.70)
n_val = int(len(patients) * 0.15)

train_patients = set(patients[:n_train])
val_patients = set(patients[n_train:n_train + n_val])

train_df = df[df['patient_id'].isin(train_patients)]
val_df = df[df['patient_id'].isin(val_patients)]

print("="*70)
print("SPLIT ANALYSIS")
print("="*70)

print(f"\nTrain: {len(train_df)} images")
print(f"  Malignant: {(train_df['label'] == 1).sum()} ({(train_df['label'] == 1).sum()/len(train_df)*100:.1f}%)")
print(f"  Benign: {(train_df['label'] == 0).sum()} ({(train_df['label'] == 0).sum()/len(train_df)*100:.1f}%)")

print(f"\nVal: {len(val_df)} images")
print(f"  Malignant: {(val_df['label'] == 1).sum()} ({(val_df['label'] == 1).sum()/len(val_df)*100:.1f}%)")
print(f"  Benign: {(val_df['label'] == 0).sum()} ({(val_df['label'] == 0).sum()/len(val_df)*100:.1f}%)")

# Check if val has enough positive samples
val_positives = (val_df['label'] == 1).sum()
if val_positives < 10:
    print(f"\n❌ CRITICAL: Only {val_positives} malignant samples in validation!")
    print(f"   Too few to compute reliable metrics!")
    print(f"   AUROC/PR-AUC will be unstable!")

print("="*70)

## 5. CHECK: What is BI-RADS 4 vs BI-RADS 5?

In [ ]:
print("BI-RADS Categories Explained:")
print("  BI-RADS 1: Negative (definitely benign)")
print("  BI-RADS 2: Benign finding (definitely benign)")
print("  BI-RADS 3: Probably benign (2% cancer risk)")
print("  BI-RADS 4: Suspicious (2-95% cancer risk) ← YOUR 'MALIGNANT'")
print("  BI-RADS 5: Highly suspicious (>95% cancer risk) ← YOUR 'MALIGNANT'")
print("  BI-RADS 6: Known biopsy-proven malignancy")

print("\n⚠️  WAIT... Are you using BI-RADS 4 as 'malignant'?")
print("   BI-RADS 4 can be 2-95% cancer risk!")
print("   Many BI-RADS 4 cases are actually BENIGN after biopsy!")

birads_4_count = (df['breast_birads'] == 'BI-RADS 4').sum()
birads_5_count = (df['breast_birads'] == 'BI-RADS 5').sum()

print(f"\nYour dataset:")
print(f"  BI-RADS 4: {birads_4_count} images (suspicious, not confirmed)")
print(f"  BI-RADS 5: {birads_5_count} images (highly suspicious)")

if birads_4_count > 0:
    print(f"\n⚠️  ISSUE: You're treating ALL BI-RADS 4 as malignant")
    print(f"   But some may actually be benign!")
    print(f"   This creates NOISY LABELS!")
    print(f"   Model can't learn reliably!")

## 6. THE REAL PROBLEM

In [ ]:
print("="*70)
print("LIKELY ROOT CAUSE")
print("="*70)

print("\nYour stratified dataset has:")
print(f"  - {(df['label'] == 1).sum()} 'malignant' (BI-RADS 4,5,6)")
print(f"  - {(df['label'] == 0).sum()} 'benign' (BI-RADS 1,2)")

print("\nBUT:")
print("  - BI-RADS 4 is SUSPICIOUS, not confirmed malignant")
print("  - Only BI-RADS 5-6 are highly likely malignant")
print("  - You're training on NOISY LABELS")

print("\nWHY MODEL CAN'T LEARN:")
print("  1. Many 'malignant' samples are actually benign")
print("  2. Model sees contradictory examples")
print("  3. Similar-looking images have different labels")
print("  4. Training loss decreases but validation is random")
print("  5. Model is just memorizing noise")

print("\nSOLUTIONS:")
print("  A. Only use BI-RADS 5-6 as malignant (smaller but cleaner dataset)")
print("  B. Get actual biopsy-confirmed labels")
print("  C. Treat BI-RADS 4 as uncertain/exclude it")

print("="*70)